In [2]:
# STEP 1: Simulate 50K+ E-commerce User Event Dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Step 1.1 - Define experiment parameters
N_USERS = 50000
CONTROL_SIZE = N_USERS // 2
TREATMENT_SIZE = N_USERS // 2

# Ground truth conversion rates
# Control: existing layout, Treatment: new layout with injected lift
CONTROL_CTR = 0.12        # 12% click-through rate
TREATMENT_CTR = 0.145     # 14.5% click-through rate — 2.5% absolute lift

CONTROL_CVR = 0.035       # 3.5% conversion rate
TREATMENT_CVR = 0.045     # 4.5% conversion rate — 1% absolute lift

print("Experiment Configuration")
print(f"Total users            : {N_USERS:,}")
print(f"Control group size     : {CONTROL_SIZE:,}")
print(f"Treatment group size   : {TREATMENT_SIZE:,}")
print(f"Control CTR            : {CONTROL_CTR*100:.1f}%")
print(f"Treatment CTR          : {TREATMENT_CTR*100:.1f}%")
print(f"Expected CTR lift      : {(TREATMENT_CTR - CONTROL_CTR)*100:.1f}%")
print(f"Control CVR            : {CONTROL_CVR*100:.1f}%")
print(f"Treatment CVR          : {TREATMENT_CVR*100:.1f}%")
print(f"Expected CVR lift      : {(TREATMENT_CVR - CONTROL_CVR)*100:.1f}%")

# Step 1.2 - Simulate user attributes (confounding variables)
user_ids = np.arange(1, N_USERS + 1)
groups = np.array(['control'] * CONTROL_SIZE + ['treatment'] * TREATMENT_SIZE)

# Device type — mobile users convert lower than desktop historically
devices = np.random.choice(
    ['mobile', 'desktop', 'tablet'],
    size=N_USERS,
    p=[0.55, 0.35, 0.10]
)

# Age group — different segments respond differently to UI changes
age_groups = np.random.choice(
    ['18-24', '25-34', '35-44', '45-54', '55+'],
    size=N_USERS,
    p=[0.20, 0.30, 0.25, 0.15, 0.10]
)

# Geography
geography = np.random.choice(
    ['North', 'South', 'East', 'West'],
    size=N_USERS,
    p=[0.25, 0.25, 0.25, 0.25]
)

# Step 1.3 - Simulate click-through outcomes
# Bernoulli trial — 1 = clicked, 0 = did not click
clicked = np.where(
    groups == 'control',
    np.random.binomial(1, CONTROL_CTR, N_USERS),
    np.random.binomial(1, TREATMENT_CTR, N_USERS)
)

# Step 1.4 - Simulate conversion outcomes
# User can only convert if they clicked first
converted = np.where(
    groups == 'control',
    np.random.binomial(1, CONTROL_CVR, N_USERS),
    np.random.binomial(1, TREATMENT_CVR, N_USERS)
)

# Enforce funnel logic — conversion requires click
converted = np.where(clicked == 1, converted, 0)

# Step 1.5 - Simulate session duration in seconds
# Treatment group sessions are slightly longer due to better UI engagement
session_duration = np.where(
    groups == 'control',
    np.random.normal(loc=180, scale=60, size=N_USERS).clip(10, 600),
    np.random.normal(loc=210, scale=65, size=N_USERS).clip(10, 600)
)

# Step 1.6 - Simulate timestamps over a 30-day experiment window
start_date = pd.Timestamp('2024-01-01')
end_date = pd.Timestamp('2024-01-31')
timestamps = pd.to_datetime(
    np.random.uniform(start_date.value, end_date.value, N_USERS)
)

# Step 1.7 - Assemble into structured DataFrame
df = pd.DataFrame({
    'user_id'          : user_ids,
    'group'            : groups,
    'device'           : devices,
    'age_group'        : age_groups,
    'geography'        : geography,
    'clicked'          : clicked,
    'converted'        : converted,
    'session_duration' : session_duration.round(1),
    'timestamp'        : timestamps
})

df = df.sort_values('timestamp').reset_index(drop=True)

# Step 1.8 - Validate dataset
print(f"\nDataset Shape: {df.shape}")
print(f"\nSchema")
print(df.dtypes)
print(f"\nSample Records (first 5)")
print(df.head())

# Step 1.9 - Group-level summary statistics
print(f"\nGroup Level Summary")
summary = df.groupby('group').agg(
    total_users      = ('user_id', 'count'),
    total_clicks     = ('clicked', 'sum'),
    total_conversions= ('converted', 'sum'),
    ctr              = ('clicked', 'mean'),
    cvr              = ('converted', 'mean'),
    avg_session_dur  = ('session_duration', 'mean')
).round(4)
print(summary)

Experiment Configuration
Total users            : 50,000
Control group size     : 25,000
Treatment group size   : 25,000
Control CTR            : 12.0%
Treatment CTR          : 14.5%
Expected CTR lift      : 2.5%
Control CVR            : 3.5%
Treatment CVR          : 4.5%
Expected CVR lift      : 1.0%

Dataset Shape: (50000, 9)

Schema
user_id                      int64
group                       object
device                      object
age_group                   object
geography                   object
clicked                      int64
converted                    int64
session_duration           float64
timestamp           datetime64[ns]
dtype: object

Sample Records (first 5)
   user_id      group   device age_group geography  clicked  converted  \
0    24252    control   tablet     25-34      East        0          0   
1    36148  treatment   mobile     25-34     North        0          0   
2    10147    control  desktop     45-54     North        0          0   
3    27635 